In [5]:
# ODIN EXECUTIVE SUITE v9.0 (AUTONOMOUS AGENCY)
# Features: Auto-Draft Emails, Monte Carlo Scenarios, & Smart Procurement.

import os
import io
import xmlrpc.client
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from typing import Dict, Any, Optional, List
from datetime import datetime, timedelta
from dataclasses import dataclass

# Environment
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, 
    TableStyle, PageBreak, Image as RLImage
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

# ================================================================
# 1. CONFIGURATION
# ================================================================
load_dotenv()
plt.switch_backend('Agg')

class Config:
    ODOO_URL = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
    ODOO_DB = os.getenv('ODOO_DB', 'vendyltd')
    ODOO_USER = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
    ODOO_PASSWORD = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')
    LLM_MODEL = "gpt-4-turbo"
    REPORT_DIR = "./odin_reports"
    
    @classmethod
    def ensure_dirs(cls):
        os.makedirs(cls.REPORT_DIR, exist_ok=True)

Config.ensure_dirs()

# ================================================================
# 2. UTILS & MODELS
# ================================================================
@dataclass
class AgentResult:
    data: Dict[str, Any]
    timestamp: datetime
    success: bool
    error: Optional[str] = None

class SafetyUtils:
    @staticmethod
    def safe_df(records: List[Dict], required_cols: List[str]) -> pd.DataFrame:
        if not records: return pd.DataFrame(columns=required_cols)
        try:
            df = pd.DataFrame(records)
            for col in required_cols:
                if col not in df.columns: df[col] = 0
            return df
        except Exception:
            return pd.DataFrame(columns=required_cols)

class OdooClient:
    def __init__(self):
        self.connect()
    
    def connect(self):
        try:
            common = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/common")
            self.uid = common.authenticate(Config.ODOO_DB, Config.ODOO_USER, Config.ODOO_PASSWORD, {})
            self.models = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/object")
            self.connected = bool(self.uid)
            print("✅ Connected to Odoo" if self.connected else "❌ Authentication failed")
        except Exception as e:
            print(f"❌ Odoo connection failed: {e}")
            self.connected = False
    
    def read(self, model: str, domain: List, fields: List) -> List[Dict]:
        if not self.connected: return []
        return self.models.execute_kw(
            Config.ODOO_DB, self.uid, Config.ODOO_PASSWORD,
            model, 'search_read', [domain], {'fields': fields}
        )

odoo = OdooClient()

# ================================================================
# 3. SCENARIO & VISUALIZATION ENGINE (Updated)
# ================================================================
class VisualizerEngine:
    @staticmethod
    def _save_plot():
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
        plt.close()
        buf.seek(0)
        return buf

    @staticmethod
    def plot_scenario_runway(current_cash, burn_rate, months=6):
        """
        Plots 3 timelines:
        1. Base Case (Current Burn)
        2. Optimistic (Burn - 20% via cost cutting)
        3. Doomsday (Revenue stops, Burn continues)
        """
        months_arr = range(months + 1)
        
        # Scenarios
        base_burn = burn_rate if burn_rate > 0 else 1000
        opt_burn = base_burn * 0.8  # 20% efficiency gain
        doom_burn = base_burn * 1.5 # Costs rise or hidden liabilities hit
        
        y_base = [current_cash - (base_burn * m) for m in months_arr]
        y_opt = [current_cash - (opt_burn * m) for m in months_arr]
        y_doom = [current_cash - (doom_burn * m) for m in months_arr]
        
        plt.figure(figsize=(6, 3.5))
        
        # Plot lines
        plt.plot(months_arr, y_opt, linestyle='--', color='green', label='Optimistic (Eff. +20%)')
        plt.plot(months_arr, y_base, linewidth=2, color='blue', label='Base Case')
        plt.plot(months_arr, y_doom, linestyle=':', color='red', label='Doomsday (Costs +50%)')
        
        plt.axhline(0, color='black', linewidth=1)
        plt.title(f'STRESS TEST SIMULATOR (Cash Runway)', fontsize=10, fontweight='bold')
        plt.xlabel('Months')
        plt.ylabel('Projected Cash ($)')
        plt.legend(fontsize=8)
        plt.grid(True, alpha=0.3)
        
        return VisualizerEngine._save_plot()

    @staticmethod
    def plot_profit_paradox(profit, cash_flow):
        labels = ['Paper Profit', 'Real Cash Flow']
        values = [profit, cash_flow]
        colors_list = ['#2ecc71', '#e74c3c'] if cash_flow < profit else ['#2ecc71', '#27ae60']
        plt.figure(figsize=(5, 3))
        bars = plt.bar(labels, values, color=colors_list, width=0.6)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height, f'${height:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        plt.title('THE PROFIT PARADOX', fontsize=10, fontweight='bold')
        plt.axhline(0, color='black', linewidth=0.8)
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        return VisualizerEngine._save_plot()

# ================================================================
# 4. AGENTS (New: Collections & Procurement)
# ================================================================

class CollectionsAgent:
    """The 'Bad Cop': Auto-writes legal demand letters."""
    def __init__(self):
        self.llm = ChatOpenAI(model=Config.LLM_MODEL, temperature=0.2)
    
    def execute(self, debtors: Dict) -> AgentResult:
        if not debtors:
            return AgentResult({"draft_email": "No debtors requiring action."}, datetime.now(), True)
            
        # Pick largest debtor
        target_name = max(debtors, key=debtors.get)
        amount = debtors[target_name]
        
        prompt = f"""Write a strict, professional collections email for a CFO.
        Debtor: {target_name}
        Amount Owed: ${amount:,.2f}
        Days Overdue: >60
        Tone: Firm, Legal, Final Warning.
        
        Output only the email body. Start with 'Subject: URGENT'."""
        
        try:
            res = self.llm.invoke(prompt)
            return AgentResult({
                "target": target_name,
                "amount": amount,
                "email_draft": res.content
            }, datetime.now(), True)
        except Exception as e:
            return AgentResult({}, datetime.now(), False, str(e))

class ProcurementAgent:
    """The 'Supply Chain' Fixer: Calculates orders for Ghost Assets."""
    def execute(self, ghost_items: List[Dict]) -> AgentResult:
        if not ghost_items:
            return AgentResult({"plan": []}, datetime.now(), True)
            
        # Logic: If stock is -5, order 10 (to reach +5 buffer)
        plan = []
        for item in ghost_items:
            # item is simple dict from Odoo
            curr = item.get('quantity', 0)
            if curr < 0:
                needed = abs(curr) + 10 # Buffer
                plan.append({
                    "product": f"Product ID {item.get('product_id', 'Unknown')}",
                    "current": curr,
                    "order_qty": needed,
                    "reason": "Ghost Asset Correction"
                })
        
        return AgentResult({"plan": plan}, datetime.now(), True)

# --- Standard Agents (Simplified for v9) ---
class AuditorAgent:
    def execute(self, d_from, d_to) -> AgentResult:
        sales = odoo.read('sale.order', [('date_order', '>=', d_from), ('date_order', '<=', d_to), ('state', 'in', ['sale', 'done'])], ['amount_total'])
        invoices = odoo.read('account.move', [('date', '>=', d_from), ('date', '<=', d_to), ('move_type', '=', 'out_invoice'), ('state', '=', 'posted')], ['amount_total'])
        
        total_sales = sum(x['amount_total'] for x in sales)
        total_invoiced = sum(x['amount_total'] for x in invoices)
        
        quants = odoo.read('stock.quant', [], ['quantity', 'product_id'])
        ghost_items = [x for x in quants if x['quantity'] < 0]
        
        return AgentResult({
            "revenue_gap": total_invoiced - total_sales,
            "ghost_items": ghost_items, # Pass full objects to Procurement
            "ghost_count": len(ghost_items)
        }, datetime.now(), True)

class FinancialistAgent:
    def execute(self, d_from, d_to) -> AgentResult:
        moves = odoo.read('account.move', [('state', '=', 'posted'), ('date', '>=', d_from), ('date', '<=', d_to)], ['amount_total', 'move_type', 'payment_state', 'partner_id', 'amount_residual'])
        df = SafetyUtils.safe_df(moves, ['amount_total', 'move_type', 'payment_state', 'amount_residual'])
        
        rev = df[df['move_type'] == 'out_invoice']['amount_total'].sum()
        exp = df[df['move_type'] == 'in_invoice']['amount_total'].sum()
        cash_flow = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum() - df[(df['move_type'] == 'in_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
        
        # Debtors
        debtors = {}
        for m in moves:
            if m['move_type'] == 'out_invoice' and m['payment_state'] != 'paid' and m['partner_id']:
                name = m['partner_id'][1]
                debtors[name] = debtors.get(name, 0) + m['amount_residual']
                
        return AgentResult({
            "profit": rev - exp,
            "cash_flow": cash_flow,
            "burn_rate": exp / 3,
            "debtors": debtors,
            "cash_balance": cash_flow # Proxy for current bank
        }, datetime.now(), True)

# ================================================================
# 5. ORCHESTRATOR
# ================================================================
class WorkflowOrchestrator:
    def __init__(self):
        self.auditor = AuditorAgent()
        self.finance = FinancialistAgent()
        self.collections = CollectionsAgent()
        self.procurement = ProcurementAgent()
        
    def execute(self, start, end):
        print("🔍 Auditor Scanning...")
        audit = self.auditor.execute(start, end)
        
        print("💸 Finance Analyzing...")
        fin = self.finance.execute(start, end)
        
        print("⚖️ Collections Agent Drafting Legal...")
        # Pass full debtor list to collections
        collect = self.collections.execute(fin.data['debtors'])
        
        print("📦 Procurement Planning Orders...")
        # Pass ghost items list to procurement
        procure = self.procurement.execute(audit.data['ghost_items'])
        
        return {
            "audit": audit.data,
            "finance": fin.data,
            "collections": collect.data,
            "procurement": procure.data,
            "period": f"{start} to {end}"
        }

# ================================================================
# 6. REPORT GENERATOR
# ================================================================
class ReportGenerator:
    def __init__(self):
        self.viz = VisualizerEngine()
        self.styles = getSampleStyleSheet()
        
    def generate(self, data, filename):
        doc = SimpleDocTemplate(filename, pagesize=letter, topMargin=40)
        story = []
        
        fin = data['finance']
        col = data['collections']
        pro = data['procurement']
        
        # Styles
        h1 = ParagraphStyle('H1', parent=self.styles['Heading1'], fontSize=20, textColor=colors.navy, spaceAfter=20)
        h2 = ParagraphStyle('H2', parent=self.styles['Heading2'], fontSize=14, textColor=colors.darkred, spaceBefore=15, borderBottomWidth=1, borderColor=colors.grey)
        code_style = ParagraphStyle('Code', parent=self.styles['Code'], fontSize=8, backColor=colors.whitesmoke, borderPadding=5)
        
        # HEADER
        story.append(Paragraph("ODIN v9.0: AUTONOMOUS ERP AGENT", h1))
        
        # 1. SCENARIO SIMULATION
        story.append(Paragraph("1. FINANCIAL FORECAST & STRESS TEST", h2))
        story.append(Spacer(1, 10))
        img_sim = RLImage(self.viz.plot_scenario_runway(fin['cash_balance'], fin['burn_rate']), width=5.5*inch, height=3*inch)
        story.append(img_sim)
        story.append(Spacer(1, 15))
        
        # 2. AUTO-ACTION: COLLECTIONS
        story.append(Paragraph("2. AUTO-ACTION: COLLECTIONS DRAFT", h2))
        if col.get('target'):
            story.append(Paragraph(f"<b>Target Debtor:</b> {col['target']} | <b>Owed:</b> ${col['amount']:,.2f}", self.styles['Normal']))
            story.append(Spacer(1, 5))
            story.append(Paragraph("<b>AI GENERATED LEGAL DRAFT (Ready to Send):</b>", self.styles['Normal']))
            email_text = col['email_draft'].replace('\n', '<br/>')
            story.append(Paragraph(email_text, code_style))
        else:
            story.append(Paragraph("No outstanding debts requiring action.", self.styles['Normal']))
            
        # 3. AUTO-ACTION: PROCUREMENT
        story.append(Paragraph("3. AUTO-ACTION: INVENTORY REBALANCING", h2))
        plan = pro.get('plan', [])
        if plan:
            story.append(Paragraph("<b>Ghost Asset Recovery Plan (Purchase Orders):</b>", self.styles['Normal']))
            table_data = [['Product', 'Current (Neg)', 'Order Qty']]
            for p in plan:
                table_data.append([p['product'], str(p['current']), str(p['order_qty'])])
            
            t = Table(table_data, colWidths=[250, 100, 100])
            t.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), colors.navy),
                ('TEXTCOLOR', (0,0), (-1,0), colors.white),
                ('GRID', (0,0), (-1,-1), 1, colors.lightgrey),
            ]))
            story.append(Spacer(1, 10))
            story.append(t)
        else:
            story.append(Paragraph("Inventory levels are healthy.", self.styles['Normal']))
            
        doc.build(story)
        return filename

if __name__ == "__main__":
    print("🚀 ODIN v9.0 Autonomous Agent - Starting...")
    orc = WorkflowOrchestrator()
    rep = ReportGenerator()
    
    end = datetime.now().strftime('%Y-%m-%d')
    start = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
    
    res = orc.execute(start, end)
    fpath = f"{Config.REPORT_DIR}/ODIN_Autonomous_Report.pdf"
    rep.generate(res, fpath)
    print(f"✅ ACTIONABLE REPORT READY: {fpath}")

✅ Connected to Odoo
🚀 ODIN v9.0 Autonomous Agent - Starting...
🔍 Auditor Scanning...
💸 Finance Analyzing...
⚖️ Collections Agent Drafting Legal...
📦 Procurement Planning Orders...
✅ ACTIONABLE REPORT READY: ./odin_reports/ODIN_Autonomous_Report.pdf
